# Data Exploration for Multimodal Foundation Models

This notebook explores the datasets used for training multimodal vision-language models including MS-COCO and LLaVA instruction data.

## Objectives
1. Load and examine MS-COCO dataset structure
2. Analyze image and caption statistics
3. Visualize data distributions
4. Explore LLaVA instruction following dataset
5. Generate insights for data preprocessing strategy

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configuration
DATA_DIR = Path('../data')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

print("Environment setup complete!")

## 1. Dataset Loading and Basic Statistics

In [ ]:
# Load datasets using our data loaders
from src.data.dataloader import COCODataset, LLaVADataset
from datasets import load_dataset

# Load Flickr30k as COCO alternative (publicly available)
try:
    flickr_dataset = load_dataset("nlphuji/flickr30k", split="train[:1000]")  # Sample for exploration
    print(f"Loaded {len(flickr_dataset)} samples from Flickr30k")
    
    # Convert to our format
    sample_data = []
    for i, item in enumerate(flickr_dataset):
        if i >= 100:  # Limit for notebook
            break
        sample_data.append({
            'image': item['image'],
            'caption': item['caption'][0] if isinstance(item['caption'], list) else item['caption'],
            'id': i
        })
    
    print(f"Prepared {len(sample_data)} samples for analysis")
    
except Exception as e:
    print(f"Error loading dataset: {e}")
    # Create dummy data for demonstration
    sample_data = []
    for i in range(50):
        sample_data.append({
            'image': Image.new('RGB', (224, 224), color=(100+i*3, 150, 200)),
            'caption': f"A sample image with various objects and scenes, example {i}",
            'id': i
        })
    print(f"Created {len(sample_data)} dummy samples for analysis")

## 2. Image Analysis

In [ ]:
# Analyze image properties
image_stats = {
    'width': [],
    'height': [],
    'aspect_ratio': [],
    'area': []
}

for item in sample_data[:50]:  # Sample subset
    img = item['image']
    width, height = img.size
    
    image_stats['width'].append(width)
    image_stats['height'].append(height)
    image_stats['aspect_ratio'].append(width / height)
    image_stats['area'].append(width * height)

# Convert to DataFrame
img_df = pd.DataFrame(image_stats)

# Display statistics
print("Image Statistics:")
print(img_df.describe())

In [ ]:
# Visualize image statistics
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Width distribution
axes[0, 0].hist(img_df['width'], bins=20, alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Image Width Distribution')
axes[0, 0].set_xlabel('Width (pixels)')
axes[0, 0].set_ylabel('Frequency')

# Height distribution
axes[0, 1].hist(img_df['height'], bins=20, alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Image Height Distribution')
axes[0, 1].set_xlabel('Height (pixels)')
axes[0, 1].set_ylabel('Frequency')

# Aspect ratio distribution
axes[1, 0].hist(img_df['aspect_ratio'], bins=20, alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Aspect Ratio Distribution')
axes[1, 0].set_xlabel('Width/Height Ratio')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(1.0, color='red', linestyle='--', label='Square (1:1)')
axes[1, 0].legend()

# Width vs Height scatter
axes[1, 1].scatter(img_df['width'], img_df['height'], alpha=0.6)
axes[1, 1].set_title('Width vs Height Scatter Plot')
axes[1, 1].set_xlabel('Width (pixels)')
axes[1, 1].set_ylabel('Height (pixels)')
axes[1, 1].plot([0, max(img_df['width'])], [0, max(img_df['width'])], 'r--', alpha=0.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'image_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Caption Analysis

In [ ]:
# Analyze caption properties
captions = [item['caption'] for item in sample_data]

caption_stats = {
    'length': [len(caption) for caption in captions],
    'word_count': [len(caption.split()) for caption in captions],
    'sentence_count': [len(caption.split('.')) for caption in captions]
}

caption_df = pd.DataFrame(caption_stats)

print("Caption Statistics:")
print(caption_df.describe())

# Most common words
from collections import Counter
import re

# Tokenize and clean
all_words = []
for caption in captions:
    words = re.findall(r'\b\w+\b', caption.lower())
    all_words.extend(words)

word_freq = Counter(all_words)
print(f"\nTotal unique words: {len(word_freq)}")
print("\nMost common words:")
for word, count in word_freq.most_common(20):
    print(f"{word}: {count}")

In [ ]:
# Visualize caption statistics
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Caption length distribution
axes[0, 0].hist(caption_df['length'], bins=20, alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Caption Character Length Distribution')
axes[0, 0].set_xlabel('Characters')
axes[0, 0].set_ylabel('Frequency')

# Word count distribution
axes[0, 1].hist(caption_df['word_count'], bins=20, alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Caption Word Count Distribution')
axes[0, 1].set_xlabel('Words')
axes[0, 1].set_ylabel('Frequency')

# Word frequency plot (top 20)
top_words = dict(word_freq.most_common(15))
axes[1, 0].bar(range(len(top_words)), list(top_words.values()))
axes[1, 0].set_title('Top 15 Most Frequent Words')
axes[1, 0].set_xlabel('Words')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_xticks(range(len(top_words)))
axes[1, 0].set_xticklabels(list(top_words.keys()), rotation=45)

# Length vs word count correlation
axes[1, 1].scatter(caption_df['word_count'], caption_df['length'], alpha=0.6)
axes[1, 1].set_title('Word Count vs Character Length')
axes[1, 1].set_xlabel('Word Count')
axes[1, 1].set_ylabel('Character Length')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'caption_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Sample Visualization

In [ ]:
# Display sample images with captions
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for i in range(min(9, len(sample_data))):
    item = sample_data[i]
    
    # Display image
    axes[i].imshow(item['image'])
    axes[i].set_title(f"ID: {item['id']}")
    axes[i].axis('off')
    
    # Add caption as text below image
    caption = item['caption'][:100] + '...' if len(item['caption']) > 100 else item['caption']
    axes[i].text(0.5, -0.1, caption, transform=axes[i].transAxes, 
                ha='center', va='top', fontsize=8, wrap=True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sample_images_captions.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Data Quality Assessment

In [ ]:
# Assess data quality
quality_checks = {
    'total_samples': len(sample_data),
    'images_without_captions': 0,
    'captions_without_images': 0,
    'duplicate_captions': 0,
    'very_short_captions': 0,
    'very_long_captions': 0,
    'invalid_images': 0
}

caption_set = set()

for item in sample_data:
    # Check for missing captions
    if not item.get('caption') or len(item['caption'].strip()) == 0:
        quality_checks['images_without_captions'] += 1
    
    # Check for missing images
    if item.get('image') is None:
        quality_checks['captions_without_images'] += 1
    
    # Check caption length
    if item.get('caption'):
        caption = item['caption']
        if len(caption.split()) < 3:
            quality_checks['very_short_captions'] += 1
        elif len(caption.split()) > 50:
            quality_checks['very_long_captions'] += 1
        
        # Check for duplicates
        if caption in caption_set:
            quality_checks['duplicate_captions'] += 1
        else:
            caption_set.add(caption)
    
    # Check image validity
    try:
        if item.get('image'):
            img = item['image']
            if img.size[0] < 32 or img.size[1] < 32:  # Very small images
                quality_checks['invalid_images'] += 1
    except Exception:
        quality_checks['invalid_images'] += 1

# Print quality report
print("Data Quality Assessment:")
print("=" * 40)
for check, count in quality_checks.items():
    percentage = (count / quality_checks['total_samples']) * 100
    print(f"{check.replace('_', ' ').title()}: {count} ({percentage:.2f}%)")

# Calculate data quality score
issues = sum(quality_checks.values()) - quality_checks['total_samples']
quality_score = max(0, (quality_checks['total_samples'] - issues) / quality_checks['total_samples'] * 100)
print(f"\nOverall Data Quality Score: {quality_score:.2f}%")

## 6. Preprocessing Strategy Recommendations

In [ ]:
# Generate preprocessing recommendations based on analysis
recommendations = []

# Image preprocessing recommendations
avg_width = np.mean(img_df['width'])
avg_height = np.mean(img_df['height'])
avg_aspect_ratio = np.mean(img_df['aspect_ratio'])

recommendations.append(f"Recommended image resize target: {int(avg_width)}x{int(avg_height)}")
recommendations.append(f"Average aspect ratio: {avg_aspect_ratio:.2f} (consider padding vs cropping)")

if avg_aspect_ratio > 1.5 or avg_aspect_ratio < 0.75:
    recommendations.append("Consider aspect ratio normalization or adaptive cropping")

# Caption preprocessing recommendations
avg_words = np.mean(caption_df['word_count'])
max_words = np.max(caption_df['word_count'])

recommendations.append(f"Average caption length: {avg_words:.1f} words")
recommendations.append(f"Recommended max sequence length: {min(512, max_words + 50)}")

if quality_checks['very_short_captions'] > 0:
    recommendations.append("Consider filtering captions with <3 words")

if quality_checks['duplicate_captions'] > 0:
    recommendations.append("Remove duplicate captions to prevent overfitting")

# Data augmentation recommendations
if np.std(img_df['width']) > avg_width * 0.3:
    recommendations.append("High image size variance - use strong augmentation")

print("Preprocessing Strategy Recommendations:")
print("=" * 50)
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

# Save recommendations to file
with open(FIGURES_DIR / 'preprocessing_recommendations.txt', 'w') as f:
    f.write("Preprocessing Strategy Recommendations\n")
    f.write("=" * 50 + "\n")
    for i, rec in enumerate(recommendations, 1):
        f.write(f"{i}. {rec}\n")

## 7. Data Splits and Statistics Summary

In [ ]:
# Create summary statistics table
summary_stats = pd.DataFrame({
    'Metric': [
        'Total Samples',
        'Average Image Width',
        'Average Image Height', 
        'Average Aspect Ratio',
        'Average Caption Length (chars)',
        'Average Caption Length (words)',
        'Unique Words',
        'Data Quality Score'
    ],
    'Value': [
        len(sample_data),
        f"{np.mean(img_df['width']):.0f}",
        f"{np.mean(img_df['height']):.0f}",
        f"{np.mean(img_df['aspect_ratio']):.2f}",
        f"{np.mean(caption_df['length']):.0f}",
        f"{np.mean(caption_df['word_count']):.1f}",
        len(word_freq),
        f"{quality_score:.1f}%"
    ]
})

print("\nDataset Summary Statistics:")
print(summary_stats.to_string(index=False))

# Save summary
summary_stats.to_csv(FIGURES_DIR / 'dataset_summary.csv', index=False)

print(f"\nAll analysis results saved to: {FIGURES_DIR}")
print("\nFiles created:")
for file in FIGURES_DIR.glob('*'):
    print(f"- {file.name}")

## Conclusions

This exploration notebook provided insights into:

1. **Image Characteristics**: Distribution of image sizes, aspect ratios, and quality
2. **Caption Properties**: Length distributions, vocabulary size, and common patterns
3. **Data Quality**: Assessment of missing data, duplicates, and anomalies
4. **Preprocessing Strategy**: Recommendations for optimal data preparation

Key findings can guide the training configuration and data augmentation strategies for optimal model performance.